# A prospective test on outside tables

`KAN_Transfer_Ledger` and `KAN_Transfer_Planning` made claims from five reaction tables: a
target that transfers (phosphines with a minimum buried volume between 25.1 and 31.4 % are the
active ones), verdicts of "does not transfer", and statements about what more molecules would
show. Every check there hid molecules of a finished table and revealed them. This notebook
scores the claims on tables from other laboratories that this series never used.

**The predictions came first.** They are in `PROSPECTIVE_TEST.md`, next to this notebook,
committed before the outside yields were read (commits 377732c4, e396464a, 0ac5f255, 087fde05
and 3105d21a on branch `claude/kan-more-datasets`). Each hypothesis is scored here by the
criterion written there, as held, failed or not testable. Where a result invites a second
look, the second look is marked *post hoc* and does not change the verdict.

| Table | What it tests |
|---|---|
| Cernak group, *JACS* 2026, 50,688 Pd-, Ni- and Cu-catalysed C-N couplings, 164 ligands (Open Reaction Database) | H1 to H6: the window, the cliff, the gate and the planning tools |
| Wang, ..., Doyle, *Nature* 2024, Pd-catalysed C-H arylation of imidazoles, 24 ligands | H7: what 24 ligands can show |
| Żurański, Gandhi, Doyle, *JACS* 2023, deoxyfluorination of 8 new alcohols | H8: does the alcohol verdict hold outside the table |
| Stevens et al., *Organometallics* 2022, Ni-catalysed borylation, 36 ligands | H9: the window in another Ni reaction |

Ligand descriptors come from the kraken library (Gensch et al., *JACS* 2022). Data are
downloaded at run time; the Cernak table comes through the Open Reaction Database API (about
50 requests, cached after the first run) and needs the `ord-schema` package
(`pip install ord-schema`). A full run takes about three and a half hours on four cores.

In [1]:
import gzip
import json
import os
import time
import urllib.request
import warnings

warnings.filterwarnings("ignore")

import importlib.metadata as imd

import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from rdkit import Chem, RDLogger
from scipy.stats import mannwhitneyu
from scipy.stats import t as student_t
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import LinearRegression

from pycse.sklearn.kan_interpret import prune_collinear
from transferledger import NewGroups, PieceModel, backtest, detectable, transfer_ledger

RDLogger.DisableLog("rdApp.*")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
N_JOBS = int(os.environ.get("PROSP_JOBS", 4))
CACHE = os.path.expanduser(os.environ.get("PROSP_CACHE", "~/.cache/pycse"))
os.makedirs(CACHE, exist_ok=True)
for p in ("numpy", "pandas", "scikit-learn", "rdkit", "transferledger", "ord-schema"):
    print(p, imd.version(p))

numpy 2.5.4
pandas 3.1.0
scikit-learn 1.9.1
rdkit 2026.9.1
transferledger 0.1.0
ord-schema 0.9.6


## Helpers

In [2]:
OCHEM = "https://raw.githubusercontent.com/doyle-lab-ucla/ochem-data/main/"
SPHERE = 4.0 / 3.0 * np.pi * 3.5**3  # kraken buried volumes are in cubic angstrom in a 3.5 A sphere
LO, HI = 25.1, 31.4  # the window found on 90 phosphines (KAN_Transfer_Ledger)
SCORE = {}  # hypothesis -> (verdict, one line)


def canon(s):
    # canonical SMILES of the largest fragment; a phosphonium salt (PCy3 HBF4) counts as its phosphine
    m = Chem.MolFromSmiles(str(s))
    if m is None:
        return None
    f = max(Chem.GetMolFrags(m, asMols=True), key=lambda x: x.GetNumHeavyAtoms())
    for a in f.GetAtoms():
        if a.GetSymbol() == "P" and a.GetFormalCharge() == 1 and a.GetTotalNumHs() > 0:
            a.SetFormalCharge(0)
            a.SetNumExplicitHs(0)
            a.SetNoImplicit(False)
    try:
        Chem.SanitizeMol(f)
    except Exception:
        return None
    return Chem.MolToSmiles(f)


def inchi_canon(s):
    m = Chem.MolFromInchi(str(s))
    return canon(Chem.MolToSmiles(m)) if m is not None else None


K = pd.read_csv(OCHEM + "kraken/kraken.csv")
K = K.assign(csmi=K["can_smiles"].map(canon)).dropna(subset=["csmi"]).drop_duplicates("csmi")
K["pct_Vbur_min"] = K["vbur_vbur_min"] * 100 / SPHERE
K = K.set_index("csmi")
print(len(K), "phosphines in kraken")

KRAKEN = {
    "pct_Vbur_min": ("vbur_vbur_min", 100.0 / SPHERE), "pct_Vbur_Boltz": ("vbur_vbur_boltz", 100.0 / SPHERE),
    "pct_Vbur_max": ("vbur_vbur_max", 100.0 / SPHERE), "Sterimol_B1": ("sterimol_B1_boltz", 1.0),
    "Sterimol_B5": ("sterimol_B5_boltz", 1.0), "Sterimol_L": ("sterimol_L_boltz", 1.0),
    "E_HOMO": ("fmo_e_homo_boltz", 1.0), "E_LUMO": ("fmo_e_lumo_boltz", 1.0), "V_min": ("vmin_vmin_boltz", 1.0),
    "NBO_charge_P": ("nbo_P_boltz", 1.0), "pyramidalization_P": ("pyr_P_boltz", 1.0), "dipole": ("dipolemoment_boltz", 1.0),
}  # fmt: skip


def ligand_descriptors(csmi):
    # the twelve kraken descriptors of KAN_Ligand_Reactivity_Cliff, pruned at |r| > 0.85 as there
    X = pd.DataFrame({k: K.loc[csmi, c].to_numpy(float) * s for k, (c, s) in KRAKEN.items()})
    kept, _ = prune_collinear(X.to_numpy(float), list(X.columns), 0.85, keep_first=["pct_Vbur_min"])
    return X[kept]


def onehot(series, prefix):
    return (
        pd.get_dummies(series.astype(str), prefix=prefix, prefix_sep=": ").astype(float).iloc[:, 1:]
    )


def window_test(L, label):
    # L: one row per ligand with its minimum buried volume and activity
    inside = L.vbur.between(LO, HI)
    a, b = L[inside].activity, L[~inside].activity
    p = mannwhitneyu(a, b, alternative="greater").pvalue
    print(f"{label}: {inside.sum()} ligands inside the window, mean activity {a.mean():.3f}; "
          f"{(~inside).sum()} outside, {b.mean():.3f}; one-sided Mann-Whitney p = {p:.4f}")  # fmt: skip
    return a.mean(), b.mean(), p

1229 phosphines in kraken


## 1. The Cernak table

Downloaded from the Open Reaction Database. The API returns at most 1000 reaction ids per
search, so the search is split by ligand (and, where needed, by aryl halide, amine, temperature,
base and catalyst), then the reactions are downloaded 1000 at a time. Reactions run without a
ligand cannot be reached by a per-ligand search and are not needed here.

In [3]:
from ord_schema.proto import dataset_pb2, reaction_pb2

API = "https://open-reaction-database.org/api"
DID = "ord_dataset-805ad863feef48579d95d86a728035f4"
PB = os.path.join(CACHE, "cernak_cn.pb.gz")


def post(path, body, tries=6):
    for k in range(tries):
        try:
            req = urllib.request.Request(API + path, data=json.dumps(body).encode(),
                                         headers={"content-type": "application/json"})  # fmt: skip
            with urllib.request.urlopen(req, timeout=300) as r:
                return r.read()
        except urllib.error.HTTPError as e:
            if e.code == 422:
                raise RuntimeError(e.read()[:300])
            time.sleep(5 * 2**k)
        except Exception:
            time.sleep(5 * 2**k)
    raise RuntimeError(path)


def ident(v):
    inner = {"op": "eq", "path": "value", "value": {"literal": v}}
    return {"op": "exists", "path": "inputs.components",
            "where": {"op": "exists", "path": "identifiers", "where": inner}}  # fmt: skip


def temp(c):
    k = c + 273.15
    return {"op": "and", "clauses": [
        {"op": "ge", "path": "conditions.temperature.setpoint_kelvin", "value": {"literal": k - 1}},
        {"op": "le", "path": "conditions.temperature.setpoint_kelvin", "value": {"literal": k + 1}}]}  # fmt: skip


def search(clauses):
    where = {
        "op": "and",
        "clauses": [{"op": "eq", "path": "dataset_id", "value": {"literal": DID}}] + clauses,
    }
    return json.loads(post("/search", {"query": {"where": where, "limit": 1000}}))["reaction_ids"]


LEVELS = [[ident(f"E{i}") for i in range(1, 7)], [ident(f"N{i}") for i in range(1, 4)],
          [temp(t) for t in (25, 30, 60, 80, 100)], [ident(f"B{i}") for i in range(25)],
          [ident(f"C{i}") for i in range(45)]]  # fmt: skip


def walk(clauses, depth, ids):
    got = search(clauses)
    if len(got) < 1000 or depth == len(LEVELS):
        ids.update(got)
        return
    for c in LEVELS[depth]:
        walk(clauses + [c], depth + 1, ids)


if not os.path.exists(PB):
    t0, ids = time.time(), set()
    for k in range(260):
        walk([ident(f"L{k}")], 0, ids)
    full, idl = dataset_pb2.Dataset(dataset_id=DID), sorted(ids)
    for i in range(0, len(idl), 1000):
        raw = post("/download", {"reaction_ids": idl[i : i + 1000]})
        full.reactions.extend(dataset_pb2.Dataset.FromString(gzip.decompress(raw)).reactions)
    open(PB, "wb").write(gzip.compress(full.SerializeToString()))
    print(f"downloaded {len(full.reactions)} reactions in {time.time() - t0:.0f} s")
ds = dataset_pb2.Dataset.FromString(gzip.decompress(open(PB, "rb").read()))

SMI, NAME = reaction_pb2.CompoundIdentifier.SMILES, reaction_pb2.CompoundIdentifier.NAME
YIELD = reaction_pb2.ProductMeasurement.YIELD
rows = []
for r in ds.reactions:
    row = {}
    for key, inp in r.inputs.items():
        role = key.replace(" soln", "")
        for c in inp.components:
            idx = [x.value for x in c.identifiers if "ChemTypeIdx" in x.details]
            smi = [x.value for x in c.identifiers if x.type == SMI]
            nm = [x.value for x in c.identifiers if x.type == NAME]
            if idx and idx[0].startswith("S"):  # a solvent of a stock solution
                row.setdefault("solvents", set()).add(nm[0] if nm else smi[0])
                continue
            row[f"{role}_id"], row[f"{role}_smiles"] = (idx or [None])[0], (smi or [None])[0]
            row[f"{role}_name"] = (nm or [None])[0]
    t = r.conditions.temperature.setpoint
    row["temperature_C"] = t.value if t.units == 1 else t.value - 273.15
    row["yield"] = [
        m.percentage.value
        for o in r.outcomes
        for p in o.products
        for m in p.measurements
        if m.type == YIELD
    ][0]
    row["solvents"] = "+".join(sorted(row.get("solvents", [])))
    rows.append(row)
cn = pd.DataFrame(rows)
cn["metal"] = cn.catalyst_smiles.map(
    lambda s: next((m for m in ("Pd", "Ni", "Cu") if m in str(s)), "none")
)
lig = cn[["ligand_id", "ligand_smiles"]].drop_duplicates("ligand_id")
lig = lig.assign(csmi=lig.ligand_smiles.map(canon))
cn = cn.merge(lig[["ligand_id", "csmi"]], on="ligand_id", how="left")
cn["in_kraken"] = cn.csmi.isin(K.index)
cn["active"] = cn["yield"] >= 10
print(
    f"{len(cn)} reactions, {cn.ligand_id.nunique()} ligands, {cn[cn.in_kraken].csmi.nunique()} of them monodentate phosphines in kraken"
)
display(cn.groupby("metal").agg(reactions=("active", "size"), with_kraken_ligand=("in_kraken", "sum"),
                                kraken_ligands=("csmi", lambda s: s[cn.loc[s.index, "in_kraken"]].nunique())))  # fmt: skip

49632 reactions, 154 ligands, 53 of them monodentate phosphines in kraken


,reactions,with_kraken_ligand,kraken_ligands
metal,,,
Cu,15792,4064,53
Ni,12032,6560,52
Pd,10340,5476,52
none,11468,2300,53


## 2. H1 and H2: the window and the cliff

*H1*: in the Ni-catalysed reactions, ligands inside the window have a higher activity (share of
reactions with at least 10 % yield) than ligands outside it. *H2*: for Ni and for Pd, a single
cut of minimum buried volume between 28 and 34 % separates ligands above and below the median
activity with a balanced accuracy above 0.65, active below the cut for Ni and above it for Pd.

In [4]:
d = cn[cn.in_kraken & cn.metal.isin(["Ni", "Pd", "Cu"])].copy()
d["vbur"] = d.csmi.map(K["pct_Vbur_min"])
CUTS = np.round(np.arange(28, 34.01, 0.1), 1)


def best_cut(L, direction):
    hi = L.activity > L.activity.median()
    best = (0.0, None)
    for c in CUTS:
        pred = (L.vbur < c) if direction == "below" else (L.vbur >= c)
        acc = 0.5 * ((pred & hi).sum() / hi.sum() + (~pred & ~hi).sum() / (~hi).sum())
        best = max(best, (acc, c))
    return best


res = {}
for m in ("Ni", "Pd", "Cu"):
    s = d[d.metal == m]
    L = s.groupby("csmi").agg(
        vbur=("vbur", "first"), activity=("active", "mean"), n=("active", "size")
    )
    a, b, p = window_test(L, f"{m} ({len(s)} reactions, {len(L)} ligands)")
    below, above = best_cut(L, "below"), best_cut(L, "above")
    print(
        f"   best cut, active below: {below[0]:.2f} at {below[1]} %; active above: {above[0]:.2f} at {above[1]} %"
    )
    res[m] = dict(a=a, b=b, p=p, below=below, above=above)
h1 = res["Ni"]["p"] < 0.05 and res["Ni"]["a"] > res["Ni"]["b"]
h2 = res["Ni"]["below"][0] > 0.65 and res["Pd"]["above"][0] > 0.65
SCORE["H1"] = (
    "held" if h1 else "failed",
    f"Ni: activity {res['Ni']['a']:.3f} inside the window, {res['Ni']['b']:.3f} outside (p = {res['Ni']['p']:.2f})",
)
SCORE["H2"] = ("held" if h2 else "failed",
               f"Pd active above {res['Pd']['above'][1]} % (balanced accuracy {res['Pd']['above'][0]:.2f}) as predicted; "
               f"Ni also active above ({res['Ni']['above'][0]:.2f} at {res['Ni']['above'][1]} %), not below ({res['Ni']['below'][0]:.2f})")  # fmt: skip
print(SCORE["H1"], SCORE["H2"], sep="\n")

Ni (6560 reactions, 52 ligands): 19 ligands inside the window, mean activity 0.066; 33 outside, 0.087; one-sided Mann-Whitney p = 0.9260
   best cut, active below: 0.44 at 32.0 %; active above: 0.71 at 29.5 %
Pd (5476 reactions, 52 ligands): 19 ligands inside the window, mean activity 0.041; 33 outside, 0.064; one-sided Mann-Whitney p = 0.9341
   best cut, active below: 0.42 at 33.0 %; active above: 0.75 at 29.5 %
Cu (4064 reactions, 53 ligands): 20 ligands inside the window, mean activity 0.027; 33 outside, 0.065; one-sided Mann-Whitney p = 0.9215
   best cut, active below: 0.49 at 28.4 %; active above: 0.63 at 31.2 %
('failed', 'Ni: activity 0.066 inside the window, 0.087 outside (p = 0.93)')
('failed', 'Pd active above 29.5 % (balanced accuracy 0.75) as predicted; Ni also active above (0.71 at 29.5 %), not below (0.44)')


### Post hoc: how much of the activity needs a metal?

The paper reports that metal-free controls give product too (through an aryne). The table has
11,468 reactions without a metal, so the question can be asked directly: under which
conditions (base, aryl halide, temperature) does product form without any metal, and what is
left of the Ni and Cu activity under the others?

In [5]:
display(cn.groupby(["base_name", "metal"]).active.mean().unstack().round(2))
bg = (
    cn[cn.metal == "none"]
    .groupby(["base_name", "electrophile_name", "temperature_C"])
    .active.mean()
)
quiet = set(bg[bg == 0].index)
print(f"condition cells with no metal-free product: {len(quiet)} of {len(bg)}")
q = d[[k in quiet for k in zip(d.base_name, d.electrophile_name, d.temperature_C)]]
for m in ("Ni", "Pd", "Cu"):
    s = q[q.metal == m]
    print(
        f"{m}, kraken phosphines, conditions without metal-free product: {s.active.sum()} active of {len(s)} reactions"
    )

metal,Cu,Ni,Pd,none
base_name,,,,
"1-ETHYL-2,2,4,4,4-PENTAKIS(DIMETHYLAMINO)-2LAMBDA5,4LAMBDA5-CATENADI(PHOSPHAZENE)",0.00,0.00,0.07,0.00
"2-tert-Butyl-1,1,3,3-tetramethylguanidine",0.00,0.00,0.00,0.00
Cs2CO3,0.09,0.01,0.01,0.00
CsOAc,0.00,0.00,0.00,0.00
DBU,0.00,0.00,0.00,0.00
DMSO,NaN,NaN,NaN,0.08
K2CO3,0.07,0.00,0.02,0.00
K3PO4,0.05,0.00,0.02,0.00
Lithium bis(trimethylsilyl)amide,0.06,NaN,NaN,0.13


condition cells with no metal-free product: 48 of 69


Ni, kraken phosphines, conditions without metal-free product: 0 active of 3577 reactions
Pd, kraken phosphines, conditions without metal-free product: 84 active of 4067 reactions
Cu, kraken phosphines, conditions without metal-free product: 1 active of 2779 reactions


## 3. H3 to H6 on the Ni reactions

Rows: Ni reactions with a kraken phosphine; target: active (yes/no); variables: the kraken
descriptors plus indicators for catalyst, base, solvent, aryl halide, amine and temperature.
*H3*: the equation chosen for a new ligand keeps a steric descriptor and has skill above zero.
*H4*: the detection law fitted on 20 ligands predicts the rates measured on 26, 40 and 52 ligands
too low or about right (mean error at most +0.05), and asks for at least the ligands that turn out
to be enough. *H5*: skill established at about half the ligands is kept with twice as many.
*H6*: training ligands spread out in descriptor space are not better than random ones.
H4 to H6 use 24 reactions per ligand, drawn at random within each ligand, to keep the run to a
few hours (allowed in advance); H3 uses every row.

In [6]:
ni = d[d.metal == "Ni"].reset_index(drop=True)
LIG = ligand_descriptors(ni.csmi)
COND = pd.concat([onehot(ni.catalyst_name, "catalyst"), onehot(ni.base_name, "base"), onehot(ni.solvents, "solvent"),
                  onehot(ni.electrophile_name, "aryl halide"), onehot(ni.nucleophile_name, "amine"),
                  onehot(ni.temperature_C.astype(int), "temperature")], axis=1)  # fmt: skip
Xn = pd.concat([LIG, COND], axis=1)
yn = ni.active.to_numpy(float)
gn = ni.ligand_name.to_numpy()
print(Xn.shape, "ligand variables:", list(LIG.columns))
STERIC = {
    "pct_Vbur_min",
    "pct_Vbur_Boltz",
    "pct_Vbur_max",
    "Sterimol_B1",
    "Sterimol_B5",
    "Sterimol_L",
}

t0 = time.time()
ledger = transfer_ledger(Xn, yn, {"new ligand": NewGroups(gn, "new ligand")}, model=PieceModel(family="binomial"),
                         family="binomial", blocks={"ligand": list(LIG.columns), "conditions": list(COND.columns)},
                         n_repeats=2, n_jobs=N_JOBS)  # fmt: skip
print(ledger)
print(f"({time.time() - t0:.0f} s)")
chosen = PieceModel(family="binomial", select_on=NewGroups(gn, "new ligand")).fit(Xn, yn)
print(chosen.equation())
row = ledger.skill[
    (ledger.skill.rung == "equation, chosen for new ligand") & (ledger.skill.scheme == "new ligand")
].iloc[0]
steric = sorted(STERIC & set(chosen.variables_))
h3 = bool(steric) and row.lo > 0
SCORE["H3"] = ("held" if h3 else "failed",
               f"equation chosen for a new ligand: skill {row.skill:.2f} [{row.lo:.2f}, {row.hi:.2f}]; "
               f"steric descriptors kept: {', '.join(steric) or 'none'}")  # fmt: skip
print(SCORE["H3"])

(6560, 42) ligand variables: ['pct_Vbur_min', 'pct_Vbur_Boltz', 'Sterimol_B1', 'Sterimol_B5', 'Sterimol_L', 'E_HOMO', 'E_LUMO', 'V_min', 'NBO_charge_P', 'pyramidalization_P', 'dipole']


Skill: share of the baseline's loss removed, on held-out rows (5 folds x 2 repeats; 95 % intervals over units)
scheme                                         reference               new ligand
rung                                                                             
baseline (training mean)            0.000 [0.000, 0.000]     0.000 [0.000, 0.000]
equation                            0.305 [0.282, 0.327]     0.307 [0.280, 0.335]
equation, chosen for new ligand     0.304 [0.281, 0.326]     0.293 [0.265, 0.321]
curves                           -2.848 [-3.079, -2.617]  -5.588 [-6.855, -4.321]
curves, tuned for new ligand     -2.654 [-2.864, -2.443]  -1.571 [-2.263, -0.879]

Memorized: reference skill given up when variables must earn their place under the shift
            memorized     lo     hi
shift                              
new ligand      0.001 -0.012  0.014

What each variable adds to the reference equation (skill lost when it is removed)
scheme                            

pieces:
  p0 = [base: Sodium trimethylsilanolate = 1]  (0 or 1)
  p1 = [base: Lithium trimethylsilanolate = 1]  (0 or 1)
  p2 = [catalyst: Ni(stb)3 = 1]  (0 or 1)
  p3 = [temperature: 60 = 1]  (0 or 1)
  p4 = [base: DBU = 1]  (0 or 1)
  p5 = [base: K3PO4 = 1]  (0 or 1)
  p6 = [catalyst: Nickel(II) bromide = 1]  (0 or 1)
  p7 = [base: MTBD = 1]  (0 or 1)
  p8 = [catalyst: Nickel(II) Acetylacetonate = 1]  (0 or 1)
  p9 = [base: PHOSPHAZENE BASE P1-T-BU-TRIS(TETRAMETHYLENE) = 1]  (0 or 1)
  p10 = [catalyst: Ni(COD)2 = 1]  (0 or 1)
  p11 = [base: Sodium tert-butoxide = 1]  (0 or 1)
  p12 = [amine: Morpholine = 1]  (0 or 1)
  p13 = [solvent: dmso = 1]  (0 or 1)
  p14 = [catalyst: Ni(iPrMe)(hexadiene) = 1]  (0 or 1)
log-odds(y) = -0.4071 -3.016*p0 -16.27*p1 -1.518*p2 -0.03309*p3 -15.13*p4 -15.8*p5 -14.14*p6 -15.54*p7 -1.481*p8 -14.88*p9 -0.6799*p10 -2.317*p11 -1.721*p12 -0.874*p13 -0.2416*p14 +1.841*p3*p12 +4.068*p11*p13 +3.027*p11*p14
('failed', 'equation chosen for a new ligand: skill 0.29

In [7]:
rng = np.random.default_rng(0)
keep = np.sort(
    np.concatenate([rng.permutation(np.flatnonzero(gn == u))[:24] for u in np.unique(gn)])
)
Xt = Xn.iloc[keep].reset_index(drop=True)
Xt = Xt.loc[:, Xt.std() > 0]
yt, gt = yn[keep], gn[keep]
labels = np.unique(gt)
print(len(Xt), "rows,", len(labels), "ligands")

1248 rows, 52 ligands


In [8]:
SHARES, REVEAL = (0.2, 0.35, 0.5, 0.7, 0.9), [26, 40, 52]
t0 = time.time()
truth = detectable(Xt, yt, NewGroups(gt, "new ligand"), family="binomial", sizes=REVEAL, shares=SHARES,
                   n_trials=12, n_jobs=N_JOBS, random_state=99)  # fmt: skip
print(truth)
measured = truth.power.set_index(["shape", "units", "share"])["detected"]
rng = np.random.default_rng(0)
rows, needs = [], []
for trial in range(3):
    sub = np.flatnonzero(np.isin(gt, rng.choice(labels, 20, replace=False)))
    D = detectable(Xt.iloc[sub].reset_index(drop=True), yt[sub], NewGroups(gt[sub], "new ligand"), family="binomial",
                   shares=SHARES, n_trials=12, horizon=100.0, n_jobs=N_JOBS, random_state=trial)  # fmt: skip
    for shape, m, q in measured.index:
        rows.append({"trial": trial, "shape": shape, "units": m, "share": q,
                     "predicted": float(D.rate(m, q, shape)), "measured": float(measured.loc[(shape, m, q)])})  # fmt: skip
    for shape in ("trend", "peak"):
        for q in SHARES:
            needs.append({"trial": trial, "shape": shape, "share": q, "needed": D.needed(q, shape)})
check = pd.DataFrame(rows).assign(error=lambda c: c.predicted - c.measured)
display(check.groupby(["shape", "units"]).agg(predicted=("predicted", "mean"), measured=("measured", "mean"),
                                              error=("error", "mean")).round(2))  # fmt: skip
needs = pd.DataFrame(needs)
cmp = []
for (shape, q), c in truth.power.groupby(["shape", "share"]):
    hit = c[c.detected >= 0.8].units
    if len(hit):
        for v in needs[(needs["shape"] == shape) & (needs.share == q)].needed:
            cmp.append({"shape": shape, "share": q, "enough": int(hit.min()),
                        "law asked for": np.inf if v is None or pd.isna(v) else int(v)})  # fmt: skip
cmp = pd.DataFrame(cmp)
display(cmp)
me = check.error.mean()
ok = (cmp["law asked for"] >= cmp["enough"]).mean() if len(cmp) else float("nan")
h4 = me <= 0.05 and (len(cmp) == 0 or ok >= 0.5)
SCORE["H4"] = ("held" if h4 else "failed",
               f"mean error of the predicted rates {me:+.2f}; the law asked for at least the ligands that were enough in "
               f"{ok:.0%} of {len(cmp)} comparisons")  # fmt: skip
print(SCORE["H4"], f"({time.time() - t0:.0f} s)")

How often a planted property of each share was established (52 ligands in hand; 'established' means in 80 % of trials)
share        0.20  0.35  0.50  0.70  0.90
shape units                              
peak  26     0.00  0.08  0.33  0.42  0.67
      40     0.08  0.33  0.50  1.00  0.92
      52     0.25  0.75  0.58  0.92  1.00
trend 26     0.17  0.42  0.58  1.00  1.00
      40     0.33  0.58  0.83  0.92  1.00
      52     0.17  0.67  1.00  1.00  1.00

A property that captures nothing was admitted in 0% of trials (trend, 26 ligands), 0% of trials (trend, 40 ligands), 0% of trials (trend, 52 ligands), 8% of trials (peak, 26 ligands), 0% of trials (peak, 40 ligands), 0% of trials (peak, 52 ligands)

trend: with 52 ligands a property must capture at least 41% of the differences between ligands to be established
peak: with 52 ligands a property must capture at least 56% of the differences between ligands to be established

Ligands needed for a weaker property:
                30 % 50 % 70 %

predicted  measured  error
shape units                            
peak  26          0.22      0.30  -0.08
      40          0.28      0.57  -0.29
      52          0.32      0.70  -0.38
trend 26          0.59      0.63  -0.05
      40          0.68      0.73  -0.05
      52          0.73      0.77  -0.03

,shape,share,enough,law asked for
0,peak,0.7,40,558
1,peak,0.7,40,79
2,peak,0.7,40,77
3,peak,0.9,40,147
4,peak,0.9,40,22
5,peak,0.9,40,22
6,trend,0.5,40,72
7,trend,0.5,40,47
8,trend,0.5,40,49
9,trend,0.7,26,32


('held', 'mean error of the predicted rates -0.15; the law asked for at least the ligands that were enough in 53% of 15 comparisons') (7399 s)


In [9]:
t0 = time.time()
B = backtest(Xt, yt, NewGroups(gt, "new ligand"), have=22, factors=(2.0,), n_trials=8, family="binomial",
             n_jobs=N_JOBS, random_state=0)  # fmt: skip
display(B.round(3))
est = B[B.state == "established"]
if len(est):
    kept = (est.realized >= est.lo).mean()
    SCORE["H5"] = ("held" if kept >= 0.9 else "failed", f"established at 22 ligands in {len(est)} of {len(B)} checks; "
                   f"skill with 44 at or above the floor in {kept:.0%}")  # fmt: skip
else:
    SCORE["H5"] = ("not testable", f"nothing established at 22 ligands in {len(B)} checks")
print(SCORE["H5"], f"({time.time() - t0:.0f} s)")

,trial,have,factor,units,rung,state,held,lo,hi,extended,realized
0,0,22,2.0,44,"equation, chosen for new ligand",nothing yet,0.044,-0.052,0.140,0.030,0.220
1,0,22,2.0,44,"curves, tuned for new ligand",nothing yet,-1.104,-2.336,0.129,-1.512,-0.399
2,1,22,2.0,44,"equation, chosen for new ligand",nothing yet,0.076,-0.002,0.153,0.068,0.252
3,1,22,2.0,44,"curves, tuned for new ligand",nothing yet,-1.017,-1.791,-0.243,-1.592,-0.813
4,2,22,2.0,44,"equation, chosen for new ligand",nothing yet,0.042,-0.068,0.153,0.034,0.235
5,2,22,2.0,44,"curves, tuned for new ligand",nothing yet,-0.861,-1.555,-0.168,-1.954,0.263
6,3,22,2.0,44,"equation, chosen for new ligand",nothing yet,0.029,-0.196,0.254,-0.020,0.221
7,3,22,2.0,44,"curves, tuned for new ligand",nothing yet,-2.016,-3.114,-0.919,-1.782,-1.362
8,4,22,2.0,44,"equation, chosen for new ligand",nothing yet,0.113,-0.029,0.255,0.222,0.176
9,4,22,2.0,44,"curves, tuned for new ligand",nothing yet,-1.514,-2.218,-0.810,-2.822,0.368


('held', 'established at 22 ligands in 1 of 16 checks; skill with 44 at or above the floor in 100%') (2610 s)


In [10]:
Zt = Xt[[c for c in LIG.columns if c in Xt]].groupby(gt).mean().loc[labels].to_numpy()
Z = (Zt - Zt.mean(0)) / Zt.std(0)


def farthest(pool, m, rng):
    chosen = [int(rng.choice(pool))]
    dist = np.linalg.norm(Z[pool] - Z[chosen[0]], axis=1)
    while len(chosen) < m:
        k = int(np.argmax(dist))
        chosen.append(int(pool[k]))
        dist = np.minimum(dist, np.linalg.norm(Z[pool] - Z[pool[k]], axis=1))
    return np.array(chosen)


def deviance(y, p):
    p = np.clip(p, 1e-3, 1 - 1e-3)
    return float(-(y * np.log(p) + (1 - y) * np.log(1 - p)).sum())


def skill_of(train, test):
    tr, te = np.flatnonzero(np.isin(gt, train)), np.flatnonzero(np.isin(gt, test))
    if np.ptp(yt[tr]) == 0:
        return 0.0, 0.0
    m = PieceModel(family="binomial", select_on=NewGroups(gt[tr], "new ligand")).fit(
        Xt.iloc[tr].reset_index(drop=True), yt[tr]
    )
    p0 = np.full(len(te), yt[tr].mean())
    cu = np.asarray(m.source_.predict(m.design_.transform(Xt.iloc[te])), float)
    return 1 - deviance(yt[te], m.predict(Xt.iloc[te])) / deviance(yt[te], p0), 1 - deviance(
        yt[te], cu
    ) / deviance(yt[te], p0)


rng, jobs = np.random.default_rng(0), []
for draw in range(12):
    perm = rng.permutation(len(labels))
    test, pool = perm[:13], perm[13:]
    for m in (13, 20, 30):
        jobs.append(
            (m, draw, "at random", labels[rng.choice(pool, m, replace=False)], labels[test])
        )
        jobs.append((m, draw, "spread out", labels[farthest(pool, m, rng)], labels[test]))
t0 = time.time()
out = Parallel(n_jobs=N_JOBS)(delayed(skill_of)(tr, te) for *_, tr, te in jobs)
S = pd.DataFrame([{"units": m, "draw": dr, "chosen": h, "equation": e, "curves": c}
                  for (m, dr, h, _, _), (e, c) in zip(jobs, out)])  # fmt: skip
W = S.pivot_table(index=["units", "draw"], columns="chosen", values="curves")
display(S.groupby(["units", "chosen"])[["equation", "curves"]].median().round(3))
better = (W["spread out"] > W["at random"]).mean()
SCORE["H6"] = (
    "held" if better <= 0.6 else "failed",
    f"spread out better than random in {better:.0%} of draws (curves)",
)
print(SCORE["H6"], f"({time.time() - t0:.0f} s)")

equation  curves
units chosen                      
13    at random     -0.088  -0.129
      spread out     0.142  -3.240
20    at random      0.067   0.268
      spread out     0.101  -0.036
30    at random      0.183   0.365
      spread out     0.127  -4.900

('held', 'spread out better than random in 19% of draws (curves)') (283 s)


## 4. H7: what 24 ligands can show (Pd-catalysed C-H arylation of imidazoles)

The plan said that for the direct arylation ligands, 24 would not be enough to establish a
property capturing a third of the ligand-to-ligand differences. On this table of the same
reaction family, held out by ligand, the detection floor for a trend is predicted above 33 %.

In [11]:
ar = pd.read_csv(OCHEM + "deebo/aryl-scope-ligand.csv")
ar["csmi"] = ar.ligand_smiles.map(canon)
ar = ar[ar.csmi.isin(K.index)].reset_index(drop=True)
Xa = pd.concat([ligand_descriptors(ar.csmi), onehot(ar.electrophile_id, "aryl bromide"),
                onehot(ar.nucleophile_id, "imidazole")], axis=1)  # fmt: skip
print(len(ar), "reactions,", ar.ligand_name.nunique(), "kraken ligands of 24")
t0 = time.time()
Da = detectable(Xa, ar["yield"].to_numpy(float), NewGroups(ar.ligand_name.to_numpy(), "new ligand"), n_trials=16,
                n_jobs=N_JOBS, random_state=0)  # fmt: skip
print(Da)
f = Da.floor("trend")
SCORE["H7"] = (
    "held" if (f is None or f > 0.33) else "failed",
    f"floor for a trend with {ar.ligand_name.nunique()} ligands: {f:.0%} of the ligand-to-ligand differences",
)
print(SCORE["H7"], f"({time.time() - t0:.0f} s)")

1408 reactions, 22 kraken ligands of 24


How often a planted property of each share was established (22 ligands in hand; 'established' means in 80 % of trials)
share        0.20  0.35  0.50  0.70  0.90
shape units                              
peak  11     0.06  0.06  0.06  0.31  0.75
      16     0.00  0.19  0.12  0.62  0.75
      22     0.00  0.19  0.38  0.88  1.00
trend 11     0.00  0.00  0.44  0.56  0.88
      16     0.06  0.38  0.62  1.00  1.00
      22     0.25  0.50  0.81  1.00  1.00

A property that captures nothing was admitted in 0% of trials (trend, 11 ligands), 0% of trials (trend, 16 ligands), 0% of trials (trend, 22 ligands), 0% of trials (peak, 11 ligands), 6% of trials (peak, 16 ligands), 0% of trials (peak, 22 ligands)

trend: with 22 ligands a property must capture at least 49% of the differences between ligands to be established
peak: with 22 ligands a property must capture at least 67% of the differences between ligands to be established

Ligands needed for a weaker property:
               30 %          5

## 5. H8: eight new alcohols (deoxyfluorination)

Training: the 740 reactions of the 37 alcohols, with the M06-2X alcohol descriptors of the 2023
paper and the substructure flags. Test: the reactions of the new alcohols with PyFluor and PBSF
and the four bases, as that paper scored them (rows it flagged to ignore are left out; replicate
runs are averaged). *H8*: no rung that uses alcohol descriptors beats the conditions-only rung by
more than 0.10 of skill, and the equation chosen on random folds scores below it.

In [12]:
DM = "https://raw.githubusercontent.com/doyle-lab-ucla/deoxyfluorination_modeling/master/data/"
GLB = [
    "number_of_atoms",
    "molar_mass",
    "electronegativity",
    "electronic_spatial_extent",
    "hardness",
    "homo_energy",
    "lumo_energy",
    "molar_volume",
    "dipole",
    "OC_length",
    "OC_L",
    "OC_B1",
    "OC_B5",
]
CF = [
    "C_APT_charge",
    "C_Mulliken_charge",
    "C_NMR_shift",
    "C_NPA_charge",
    "C_VBur",
    "C_angle",
    "C_PVBur",
]
OF = ["O_APT_charge", "O_Mulliken_charge", "O_NMR_shift", "O_NPA_charge", "O_VBur"]  # fmt: skip
dft = pd.read_csv(DM + "alcohols_Boltzmann_M062X_THF.csv").set_index("name")[GLB + CF + OF]
FLAGS = ["primary", "secondary", "tertiary", "ring_4", "ring_5", "ring_6", "ring_7", "benzylic", "allylic",
         "homobenzylic", "homoallylic", "alpha_carbonyl", "beta_carbonyl", "hemiacetal", "alpha_amino"]  # fmt: skip
tr = pd.concat(
    [pd.read_csv(DM + "fluorination_train.csv"), pd.read_csv(DM + "fluorination_test.csv")],
    ignore_index=True,
)
tr = tr.rename(columns={"sulfonyl.fluoride": "so2f"})
tr["so2f"] = tr.so2f.str.replace(".", "-", regex=False).str.replace("3-", "4-", regex=False)
src = {"primary": "primary", "secondary": "secondary", "tertiary": "tertiary", "ring_4": "4.membered_ring",
       "ring_5": "5.membered_ring", "ring_6": "6.membered_ring", "ring_7": "7.membered_ring", "benzylic": "benzylic",
       "allylic": "allylic", "homobenzylic": "homobenzylic", "homoallylic": "homoallylic",
       "alpha_carbonyl": "alpha.carbonyl", "beta_carbonyl": "beta_carbonyl", "hemiacetal": "hemiacetal",
       "alpha_amino": "alpha.amino"}  # fmt: skip
train = pd.concat(
    [
        tr[["alcohol", "base", "so2f", "yield"]],
        pd.DataFrame({k: tr["alcohol_" + v] for k, v in src.items()}),
    ],
    axis=1,
)
vf = pd.read_csv(DM + "Deoxyfluorination_validation_alcohols.csv", encoding="utf-8-sig").set_index(
    "name"
)
vflags = pd.DataFrame({"primary": vf.order.eq("primary"), "secondary": vf.order.eq("secondary"),
                       "tertiary": vf.order.eq("tertiary"), **{f"ring_{k}": vf.ring_size.eq(k) for k in (4, 5, 6, 7)},
                       **{c: vf[c] for c in FLAGS[7:]}}).astype(float)  # fmt: skip
vy = pd.read_csv(DM + "Deoxyfluorination_validation_data.csv", encoding="utf-8-sig")
vy = vy[vy.ignore == 0].groupby(["name", "base", "so2f"], as_index=False)["yield"].mean()
val = vy.rename(columns={"name": "alcohol"}).join(vflags, on="alcohol")
dft_v = dft.copy()
dft_v.loc["1am", "C_PVBur"] = 0.425  # the 2023 paper's correction for 1am
print(
    f"training: {len(train)} reactions, {train.alcohol.nunique()} alcohols; new: {len(val)} reactions, alcohols {sorted(val.alcohol.unique())}"
)
kept, _ = prune_collinear(dft.loc[train.alcohol.unique()].to_numpy(float), list(dft.columns), 0.85,
                          keep_first=["electronegativity", "C_PVBur"])  # fmt: skip


def design(df, D):
    cond = pd.get_dummies(df[["base", "so2f"]].astype(str), prefix_sep=": ").astype(float)
    return cond, pd.concat(
        [
            cond,
            df[FLAGS].astype(float).reset_index(drop=True),
            D.loc[df.alcohol, kept].reset_index(drop=True),
        ],
        axis=1,
    )


ct, Xtr = design(train, dft)
cv, Xva = design(val, dft_v)
cv, Xva = (
    cv.reindex(columns=ct.columns, fill_value=0.0),
    Xva.reindex(columns=Xtr.columns, fill_value=0.0),
)
ytr, yva = train["yield"].to_numpy(float), val["yield"].to_numpy(float)
base = ytr.mean()
new_eq = PieceModel(
    family="gaussian", select_on=NewGroups(train.alcohol.to_numpy(), "new alcohol")
).fit(Xtr, ytr)
pred = {
    "conditions only": LinearRegression().fit(ct, ytr).predict(cv),
    "equation, chosen on random folds": PieceModel(family="gaussian").fit(Xtr, ytr).predict(Xva),
    "equation, chosen for a new alcohol": new_eq.predict(Xva),
    "curves, tuned for a new alcohol": np.asarray(new_eq.source_.predict(new_eq.design_.transform(Xva)), float),
    "gradient boosting": GradientBoostingRegressor(n_estimators=300, learning_rate=0.1, max_depth=4, min_samples_split=5,
                                                   min_samples_leaf=3, subsample=0.8, random_state=42).fit(Xtr, ytr).predict(Xva),
}  # fmt: skip
alc = val.alcohol.to_numpy()


def skill(p, m=None):
    m = np.ones(len(yva), bool) if m is None else m
    return 1 - np.sum((yva[m] - np.clip(p[m], 0, 100)) ** 2) / np.sum((yva[m] - base) ** 2)


rows = []
for k, p in pred.items():
    diff = skill(p) - skill(pred["conditions only"])
    loo = np.array(
        [skill(p, alc != a) - skill(pred["conditions only"], alc != a) for a in np.unique(alc)]
    )
    n = len(loo)
    se = np.sqrt((n - 1) / n * np.sum((loo - loo.mean()) ** 2))
    q = student_t.ppf(0.975, n - 1)
    rows.append({"rung": k, "skill on the new alcohols": skill(p), "mean absolute error": np.mean(np.abs(yva - np.clip(p, 0, 100))),
                 "minus conditions only": diff, "interval (delete one alcohol)": f"[{diff - q * se:+.2f}, {diff + q * se:+.2f}]"})  # fmt: skip
R = pd.DataFrame(rows).set_index("rung")
display(R.round(3))
print(new_eq.equation())
uses = R["skill on the new alcohols"].drop("conditions only")
h8 = (uses.max() - R.loc["conditions only", "skill on the new alcohols"] <= 0.10) and (
    R.loc["equation, chosen on random folds", "skill on the new alcohols"]
    < R.loc["conditions only", "skill on the new alcohols"]
)
SCORE["H8"] = ("held" if h8 else "failed",
               f"best rung with alcohol descriptors ({uses.idxmax()}) {uses.max() - R.loc['conditions only', 'skill on the new alcohols']:+.2f} "
               f"over conditions only, interval {R.loc[uses.idxmax(), 'interval (delete one alcohol)']}; equations below conditions only")  # fmt: skip
print(SCORE["H8"])

training: 740 reactions, 37 alcohols; new: 48 reactions, alcohols ['1al', '1am', '1an', '1ao', '1ap', '1aq']


,skill on the new alcohols,mean absolute error,minus conditions only,interval (delete one alcohol)
rung,,,,
conditions only,0.109,21.715,0.000,"[+0.00, +0.00]"
"equation, chosen on random folds",-0.113,24.549,-0.222,"[-1.06, +0.62]"
"equation, chosen for a new alcohol",-0.431,28.698,-0.540,"[-1.06, -0.02]"
"curves, tuned for a new alcohol",-0.017,23.845,-0.127,"[-0.84, +0.59]"
gradient boosting,0.220,18.492,0.110,"[-0.55, +0.77]"


pieces:
  p0 = exp(-15.3 * (C_PVBur - 0.336))
  p1 = S((dipole - 4.21) / 0.0188)   [0 below 4.21, 1 above]
  p2 = [base: DBU = 1]  (0 or 1)
  p3 = [so2f: 4-Cl = 1]  (0 or 1)
  p4 = [so2f: PyFluor = 1]  (0 or 1)
y = 22.14 +58.25*p0 +18.65*p1 +1.352*p2 -10.7*p3 -9.448*p4 -42.82*p0*p2
  where S(t) = 1 / (1 + exp(-t)), a step from 0 to 1 of width about 4 in t
('failed', 'best rung with alcohol descriptors (gradient boosting) +0.11 over conditions only, interval [-0.55, +0.77]; equations below conditions only')


## 6. H9: the window in Ni-catalysed borylation

Stevens et al. 2022: 33 aryl (pseudo)halides with 24 ligands in two alcohols, and an external set
of 12 further ligands with 2 substrates.

In [13]:
nb = pd.concat([pd.read_csv(OCHEM + "NiB/rxns/inchi.csv").assign(file="main"),
                pd.read_csv(OCHEM + "NiB/rxns/inchi_val.csv").assign(file="external")])  # fmt: skip
nb["csmi"] = nb.ligand_inchi.map(inchi_canon)
nb = nb[nb.csmi.isin(K.index)].copy()
nb["vbur"], nb["active"] = nb.csmi.map(K["pct_Vbur_min"]), nb["yield"] >= 10
Lb = nb.groupby("ligand_name").agg(
    vbur=("vbur", "first"),
    activity=("active", "mean"),
    n=("active", "size"),
    file=("file", "first"),
)
display(Lb.sort_values("vbur").round(2).T)
a, b, p = window_test(Lb, f"Ni borylation ({len(nb)} reactions, {len(Lb)} kraken ligands)")
print(f"share of all reactions active: {nb.active.mean():.2f}")
SCORE["H9"] = (
    "held" if (p < 0.05 and a > b) else "failed",
    f"activity {a:.2f} inside the window, {b:.2f} outside (p = {p:.3f}); {nb.active.mean():.0%} of all reactions are active",
)
print(SCORE["H9"])

ligand_name,PnBu3 HBF4,PEt3,PPh2Cy,PXy3,P(p-CF3-Ph)3,P(p-Anis)3,PPh3,A-paPhos,P(p-F-Ph)3,PBn3,PPhCy2,PPh2(o-Anis),Ph-SPhos,Ph-DavePhos,Cy-BippyPhos,...,CX-POMeCy,MePhos,Cy-PhenCar-Phos,CX-PCy,Cy-JohnPhos-OMe,S-Phos,VPhos,Cy-JohnPhos,CM-Phos,CPhos,P(o-Anis)3,Cy-vBRIDP,CX-FBu,PPhtBu2,Kwong
vbur,24.24,24.4,27.96,28.12,28.16,28.21,28.22,28.25,28.32,28.46,28.84,28.9,29.04,29.55,30.12,...,30.88,31.51,31.53,31.57,31.62,31.67,31.74,31.76,32.28,32.59,33.06,33.88,34.49,35.15,37.66
activity,0.87,0.5,0.97,0.92,1.0,0.95,0.94,0.95,0.95,1.0,1.0,1.0,0.92,0.91,0.91,...,1.0,0.89,0.91,0.91,1.0,1.0,0.88,0.94,1.0,1.0,0.91,0.91,0.89,0.52,0.91
n,60,4,66,66,4,66,66,66,66,4,4,4,66,66,66,...,4,66,66,66,4,4,66,66,4,4,66,66,66,66,66
file,main,external,main,main,external,main,main,main,main,external,external,external,main,main,main,...,external,main,main,main,external,external,main,main,external,external,main,main,main,main,main


Ni borylation (1494 reactions, 34 kraken ligands): 18 ligands inside the window, mean activity 0.944; 16 outside, 0.877; one-sided Mann-Whitney p = 0.0241
share of all reactions active: 0.89
('held', 'activity 0.94 inside the window, 0.88 outside (p = 0.024); 89% of all reactions are active')


## 7. Scorecard

In [14]:
card = pd.DataFrame(
    [
        {"hypothesis": k, "verdict": v[0], "what was measured": v[1]}
        for k, v in sorted(SCORE.items())
    ]
).set_index("hypothesis")
with pd.option_context("display.max_colwidth", 200):
    display(card)

,verdict,what was measured
hypothesis,,
H1,failed,"Ni: activity 0.066 inside the window, 0.087 outside (p = 0.93)"
H2,failed,"Pd active above 29.5 % (balanced accuracy 0.75) as predicted; Ni also active above (0.71 at 29.5 %), not below (0.44)"
H3,failed,"equation chosen for a new ligand: skill 0.29 [0.27, 0.32]; steric descriptors kept: none"
H4,held,mean error of the predicted rates -0.15; the law asked for at least the ligands that were enough in 53% of 15 comparisons
H5,held,established at 22 ligands in 1 of 16 checks; skill with 44 at or above the floor in 100%
H6,held,spread out better than random in 19% of draws (curves)
H7,held,floor for a trend with 22 ligands: 49% of the ligand-to-ligand differences
H8,failed,"best rung with alcohol descriptors (gradient boosting) +0.11 over conditions only, interval [-0.55, +0.77]; equations below conditions only"
H9,held,"activity 0.94 inside the window, 0.88 outside (p = 0.024); 89% of all reactions are active"


### Reading

_(written after the run)_

**Scorecard.** Five of nine predictions held, four failed. Two of the five held only formally (H4 by a rule fixed in code rather than in `PROSPECTIVE_TEST.md`, H5 on one check), and the C-N table could not have tested the window on Ni (post hoc, below).

| | Prediction | Verdict | What was measured |
|---|---|---|---|
| H1 | The window marks the active ligands in Ni C-N couplings | failed | 52 ligands: activity 0.066 inside, 0.087 outside (p = 0.93) |
| H2 | One cut: Ni active below, Pd above | failed | Pd active above 29.5 % (0.75), as predicted; Ni also active above (0.71), not below (0.44) |
| H3 | Equation for a new ligand keeps a steric descriptor, skill above zero | failed | skill 0.29 [0.27, 0.32], with no ligand descriptor; ligand block 0.000 |
| H4 | Detection law errs toward more ligands | held, by a rule not written in advance | rates too low by 0.03 to 0.05 (trend), 0.08 to 0.38 (peak); asked for enough ligands in 8 of 15 comparisons |
| H5 | Established skill is kept | held, on one check | established at 22 ligands in 1 of 16 checks; kept at 44 in that one |
| H6 | Spreading ligands out does not help | held | better than random in 19 % of draws (curves) |
| H7 | 24 arylation ligands are not enough | held | floor 49 % (trend), 67 % (peak) with 22 ligands |
| H8 | Alcohol descriptors do not carry to new alcohols | failed, by 0.01 | gradient boosting +0.11 over conditions only, interval [-0.55, +0.77]; both equations below conditions only |
| H9 | The window marks the active ligands in Ni borylation | held, weakly | 0.94 inside, 0.88 outside (p = 0.024); 89 % of all reactions active |

**The window on Ni C-N couplings (H1 to H3): failed, and the table could not have shown it.**

- *Post hoc.* The Ni activity in this table is the activity without a metal. Under the conditions where the metal-free controls give no product (48 of 69 condition cells), 0 of 3577 Ni reactions with a kraken phosphine are active, and 1 of 2779 with Cu. Pd keeps 84 of 4067 (2.1 %). The active Ni reactions sit in two bases that give product with no metal (NaHMDS: 0.55 with Ni, 0.57 without; KOtBu: 0.20 and 0.27).
- So there was no ligand-dependent Ni activity to mark. H1 and H2 failed for Ni, and the reason is in the table, not in the window: it is neither confirmed nor refuted here. This could not have been seen without reading the yields, and the verdicts stay "failed".
- The gate behaved as it should on such a table. Held out by ligand, the ligand block adds 0.000 [-0.000, +0.000] and the conditions block 0.31 [0.28, 0.34]; the equation chosen for a new ligand is made of bases, catalysts, solvent, amine and temperature. It did not report a ligand effect that is not there.
- For Pd, the direction written down from the ligation state held (active above about 29.5 %), but on 2 % of reactions outside the background; read it as a lead, not a result.

**The window in Ni borylation (H9): held, weakly.** Ligands inside the window were more often active, 0.94 against 0.88 (p = 0.024). Almost every ligand works in this reaction, so the window separates little; most of the gap in means is two ligands outside it, PPhtBu2 (0.52) and PEt3 (0.50, on 4 reactions); without them the outside mean is 0.93.

**The planning tools (H4 to H7).**

- *H7 held.* The plan said 24 direct arylation ligands would not be enough for a property of the window's strength. On the 22 kraken ligands of the bandit table a trend must capture 49 % of the differences between ligands to be found in 80 % of plantings, a peak 67 %. The window captures a third.
- *H4: the direction held, the count is mixed.* Fitted on 20 ligands, the law's rates for 26, 40 and 52 ligands were too low, as on the phosphines: by 0.03 to 0.05 for a trend (0.03 to 0.06 there) and 0.08 to 0.38 for a peak (0.07 to 0.16 there). The ligands it asked for were at least those that turned out enough in 8 of 15 comparisons. `PROSPECTIVE_TEST.md` did not say in how many; the notebook's rule (at least half) was set before the run but was not committed in advance. Under "every comparison" H4 fails. *Post hoc:* in 6 comparisons 26 ligands, the smallest size measured, were already enough, so asking for fewer cannot be shown wrong; in the other 9 the law asked for enough in 7, and twice asked for 22 where 26 were not enough (peak capturing 90 %).
- *H5 is not evidence.* The curve was established at 22 ligands in 1 of 16 checks, and kept at 44 in that one. The other 15 were "nothing yet", and there the equation's skill at 44 (0.18 to 0.27) was in 6 of 7 checks above where the curve at 22 pointed, as in `KAN_Transfer_Planning`.
- *H6 held* for the curves rung (19 % of draws), but on a table where the ligand descriptors carry nothing, how ligands are chosen by those descriptors can matter little. The curves rung was unstable here (median skill down to -4.9); the equation rung gave spread-out ligands a higher median at 13 and 20 ligands and a lower one at 30.

**New alcohols (H8): failed, by 0.01 of skill.** Gradient boosting with the alcohol descriptors scored 0.22 on the 6 scorable new alcohols against 0.11 for conditions only, +0.11 where at most +0.10 was allowed. Deleting one alcohol at a time moves that difference between -0.55 and +0.77, so six alcohols cannot separate the two. Both equations scored below conditions only (chosen on random folds -0.22, chosen for a new alcohol -0.54, interval [-1.06, -0.02]), as predicted.

**What this does not show.**

- The cleanest test of the window, 28 new ligands in the same Ni Suzuki reactions (LeSueur et al., *Eur. J. Org. Chem.* 2024), was not run: its supporting tables could not be downloaded here.
- 1,056 reactions of the Cernak table without a ligand identifier were not fetched; none of them could enter H1 to H6.
- H4 to H6 used 24 reactions per ligand, drawn at random within each ligand, as allowed in advance; H3 used every row.
- Of the 8 new alcohols, 1al was partly seen before H8 was written (disclosed in `PROSPECTIVE_TEST.md`) and 1ar has every row flagged in the source file, so 6 were scored.